In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [3]:
import requests
from io import StringIO

crash_df = pd.read_csv("Motor_Vehicle_Collisions_-_Crashes_20260204.csv")

/tmp/ipykernel_73/665299587.py:4: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  crash_df = pd.read_csv("Motor_Vehicle_Collisions_-_Crashes_20260204.csv")


In [8]:
weather_df = pd.read_excel("open-meteo-40.74N74.04W51m.xlsx")

In [7]:
import pandas as pd

base_url = "https://data.cityofnewyork.us/resource/ic3t-wcy2.csv"

limit = 50000
offset = 0
dfs = []

while True:

    url = (
        f"{base_url}?"
        "$select=job__,job_type,borough,gis_latitude,gis_longitude,pre__filing_date,signoff_date"
        f"&$limit={limit}&$offset={offset}"
    )

    df = pd.read_csv(url)

    if df.empty:
        break

    dfs.append(df)
    offset += limit

construction_df = pd.concat(dfs, ignore_index=True)

print("Final dataset shape:", construction_df.shape)


Final dataset shape: (2714469, 7)


In [14]:
construction_df.head(20)

,job__,job_type,borough,gis_latitude,gis_longitude,pre__filing_date,signoff_date
0,321386512,A1,BROOKLYN,40.676829,-73.981570,02/28/2019,06/23/2023
1,340922871,A3,BROOKLYN,40.680983,-73.963823,01/08/2026,NaN
2,301776291,A1,BROOKLYN,40.638728,-74.001250,04/16/2004,10/26/2006
3,340922880,A3,BROOKLYN,40.651060,-73.974759,01/08/2026,NaN
4,440673852,A2,QUEENS,40.669797,-73.729699,05/03/2021,02/23/2022
5,421789559,DM,QUEENS,40.727573,-73.855368,05/03/2021,NaN
6,240304167,A3,BRONX,40.807149,-73.929969,05/03/2021,NaN
7,440707709,A2,QUEENS,40.680871,-73.744992,03/08/2022,NaN
8,340819350,A3,BROOKLYN,40.618773,-74.000274,08/09/2021,NaN
9,301829822,NB,BROOKLYN,40.669336,-73.858941,02/09/2006,NaN


In [9]:
traffic_df = pd.read_csv("Automated_Traffic_Volume_Counts_20260221.csv")

/tmp/ipykernel_73/3036077650.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  traffic_df = pd.read_csv("Automated_Traffic_Volume_Counts_20260221.csv")


Clean Crash Dataset

Take the cleaning logic from: Cleaned\_MotorVehicle\_and\_AutoTraffic\_Data\.ipynb

Include:

convert CRASH DATE

convert CRASH TIME

handle missing BOROUGH

convert injury columns to numeric

filter years

In [16]:
print("Crash dataset shape:", crash_df.shape)
print("Weather dataset shape:", weather_df.shape)
print("Construction dataset shape:", construction_df.shape)
print("Traffic dataset shape:", traffic_df.shape)

Crash dataset shape: (2239103, 29)
Weather dataset shape: (100452, 13)
Construction dataset shape: (2714469, 7)
Traffic dataset shape: (1838386, 14)


In [8]:
print("CRASH DATASET INFO")
crash_df.info()

print("\nWEATHER DATASET INFO")
weather_df.info()

print("\nCONSTRUCTION DATASET INFO")
construction_df.info()

print("\nTRAFFIC DATASET INFO")
traffic_df.info()

CRASH DATASET INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2239103 entries, 0 to 2239102
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     object 
 1   CRASH TIME                     object 
 2   BOROUGH                        object 
 3   ZIP CODE                       object 
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   LOCATION                       object 
 7   ON STREET NAME                 object 
 8   CROSS STREET NAME              object 
 9   OFF STREET NAME                object 
 10  NUMBER OF PERSONS INJURED      float64
 11  NUMBER OF PERSONS KILLED       float64
 12  NUMBER OF PEDESTRIANS INJURED  int64  
 13  NUMBER OF PEDESTRIANS KILLED   int64  
 14  NUMBER OF CYCLIST INJURED      int64  
 15  NUMBER OF CYCLIST KILLED       int64  
 16  NUMBER OF MOTORIST INJURED     int64  
 17  NUMBER OF MOTORIST KILLED  

In [9]:
print("Crash columns:")
print(crash_df.columns)

print("\nWeather columns:")
print(weather_df.columns)

print("\nConstruction columns:")
print(construction_df.columns)

print("\nTraffic columns:")
print(traffic_df.columns)

Crash columns:
Index(['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE',
       'LONGITUDE', 'LOCATION', 'ON STREET NAME', 'CROSS STREET NAME',
       'OFF STREET NAME', 'NUMBER OF PERSONS INJURED',
       'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED',
       'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED',
       'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED',
       'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1',
       'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3',
       'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5',
       'COLLISION_ID', 'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2',
       'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5'],
      dtype='object')

Weather columns:
Index(['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'rain (mm)',
       'snowfall (cm)', 'cloud_cover (%)', 'wind_speed_10m (km/h)',
       'Unnamed: 7', 'Unnamed: 8', 'Unna

In [10]:
print("Crash missing values:")
print(crash_df.isnull().sum())

print("\nWeather missing values:")
print(weather_df.isnull().sum())

print("\nConstruction missing values:")
print(construction_df.isnull().sum())

print("\nTraffic missing values:")
print(traffic_df.isnull().sum())

Crash missing values:
CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           684087
ZIP CODE                          684368
LATITUDE                          240564
LONGITUDE                         240564
LOCATION                          240564
ON STREET NAME                    489372
CROSS STREET NAME                 856050
OFF STREET NAME                  1842105
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED              31
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8021
CONTRIBUTING FACTOR VEHICLE 2     361649
CONTRIBUTING FACTOR VEHICLE 3    2077021
CONTRIBUTING FACTOR VEHICLE 4    2202061
CONTRIBUTING FACTOR VEHICLE 5    2228953
COLLISION_ID                       

In [11]:
print(crash_df.dtypes)
print(weather_df.dtypes)
print(construction_df.dtypes)
print(traffic_df.dtypes)

CRASH DATE                        object
CRASH TIME                        object
BOROUGH                           object
ZIP CODE                          object
LATITUDE                         float64
LONGITUDE                        float64
LOCATION                          object
ON STREET NAME                    object
CROSS STREET NAME                 object
OFF STREET NAME                   object
NUMBER OF PERSONS INJURED        float64
NUMBER OF PERSONS KILLED         float64
NUMBER OF PEDESTRIANS INJURED      int64
NUMBER OF PEDESTRIANS KILLED       int64
NUMBER OF CYCLIST INJURED          int64
NUMBER OF CYCLIST KILLED           int64
NUMBER OF MOTORIST INJURED         int64
NUMBER OF MOTORIST KILLED          int64
CONTRIBUTING FACTOR VEHICLE 1     object
CONTRIBUTING FACTOR VEHICLE 2     object
CONTRIBUTING FACTOR VEHICLE 3     object
CONTRIBUTING FACTOR VEHICLE 4     object
CONTRIBUTING FACTOR VEHICLE 5     object
COLLISION_ID                       int64
VEHICLE TYPE COD

In [18]:
print("Crash duplicates:", crash_df.duplicated().sum())
print("Weather duplicates:", weather_df.duplicated().sum())
print("Construction duplicates:", construction_df.duplicated().sum())
print("Traffic duplicates:", traffic_df.duplicated().sum())

Crash duplicates: 0
Weather duplicates: 0
Construction duplicates: 1013792
Traffic duplicates: 0


## Convert crash date and filter years

In [11]:
crash_df["CRASH DATE"] = pd.to_datetime(crash_df["CRASH DATE"])

crash_df = crash_df[(crash_df["CRASH DATE"].dt.year >= 2015) &
                    (crash_df["CRASH DATE"].dt.year <= 2025)]



## Convert crash time and extract hour

In [13]:
crash_df["CRASH TIME"] = pd.to_datetime(
    crash_df["CRASH TIME"], format="%H:%M", errors="coerce"
)
crash_df["HOUR"] = crash_df["CRASH TIME"].dt.hour

## Create unified date column

In [14]:
# Create unified date column
crash_df["date"] = crash_df["CRASH DATE"]

## Remove rows with no location

In [16]:
crash_df = crash_df[~(
    crash_df["BOROUGH"].isna() &
    crash_df["LATITUDE"].isna() &
    crash_df["LONGITUDE"].isna()
)]

## Borough prediction using KNN

In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier

# Check missing BEFORE
print("Missing BOROUGH BEFORE:", crash_df["BOROUGH"].isna().sum())

# Work on a copy to avoid warnings
df_ml = crash_df.copy()

# Drop rows with missing coordinates
df_ml = df_ml.dropna(subset=["LATITUDE", "LONGITUDE"])

# Separate known / unknown borough
train_df = df_ml[df_ml["BOROUGH"].notna()].copy()
test_df = df_ml[df_ml["BOROUGH"].isna()].copy()

# Encode borough
le = LabelEncoder()
train_df["borough_encoded"] = le.fit_transform(train_df["BOROUGH"])

# Train model
X_train = train_df[["LATITUDE", "LONGITUDE"]]
y_train = train_df["borough_encoded"]

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

# Predict missing borough
if len(test_df) > 0:
    X_test = test_df[["LATITUDE", "LONGITUDE"]]
    predicted = model.predict(X_test)

    test_df["BOROUGH"] = le.inverse_transform(predicted)

    # Update original df_filtered
    crash_df.update(test_df)

# Check missing AFTER
print("Missing BOROUGH AFTER:", crash_df["BOROUGH"].isna().sum())

Missing BOROUGH BEFORE: 439752
Missing BOROUGH AFTER: 0


In [19]:
from sklearn.metrics import accuracy_score
X = train_df[["LATITUDE", "LONGITUDE"]]
y = train_df["borough_encoded"]

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))

Validation Accuracy: 0.9956162040068964


## Convert injury columns to numeric

In [20]:
injury_columns = [
"NUMBER OF PERSONS INJURED",
"NUMBER OF PERSONS KILLED",
"NUMBER OF PEDESTRIANS INJURED",
"NUMBER OF PEDESTRIANS KILLED",
"NUMBER OF CYCLIST INJURED",
"NUMBER OF CYCLIST KILLED",
"NUMBER OF MOTORIST INJURED",
"NUMBER OF MOTORIST KILLED"
]

for col in injury_columns:
    crash_df[col] = pd.to_numeric(crash_df[col], errors="coerce").fillna(0)

## Remove rows with missing date

In [21]:
crash_df = crash_df.dropna(subset=["date"]).reset_index(drop=True)

## Drop unnecessary columns

In [22]:
columns_to_remove = [
    "ON STREET NAME",
    "OFF STREET NAME",
    "CROSS STREET NAME",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5",
    "COLLISION_ID",
    "VEHICLE TYPE CODE 4",
    "VEHICLE TYPE CODE 5"
]

crash_df = crash_df.drop(columns=columns_to_remove, errors="ignore")

## Standardize borough names

In [23]:
crash_df["BOROUGH"] = crash_df["BOROUGH"].str.lower().str.strip()

In [25]:
print("Crash dataset shape:", crash_df.shape)

crash_df.head()

Crash dataset shape: (1599858, 23)


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,...,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,HOUR,date
0,2023-11-01,1900-01-01 01:29:00,brooklyn,11230.0,40.621790,-73.970024,"(40.62179, -73.970024)",1.0,0.0,0,...,1,0,Unspecified,Unspecified,Unspecified,Moped,Sedan,Sedan,1,2023-11-01
1,2021-09-11,1900-01-01 09:35:00,brooklyn,11208.0,40.667202,-73.866500,"(40.667202, -73.8665)",0.0,0.0,0,...,0,0,Unspecified,NaN,NaN,Sedan,NaN,NaN,9,2021-09-11
2,2021-12-14,1900-01-01 08:13:00,brooklyn,11233.0,40.683304,-73.917274,"(40.683304, -73.917274)",0.0,0.0,0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,8,2021-12-14
3,2021-12-14,1900-01-01 17:05:00,brooklyn,NaN,40.709183,-73.956825,"(40.709183, -73.956825)",0.0,0.0,0,...,0,0,Passing Too Closely,Unspecified,NaN,Sedan,Tractor Truck Diesel,NaN,17,2021-12-14
4,2021-12-14,1900-01-01 08:17:00,bronx,10475.0,40.868160,-73.831480,"(40.86816, -73.83148)",2.0,0.0,0,...,2,0,Unspecified,Unspecified,NaN,Sedan,Sedan,NaN,8,2021-12-14


In [10]:
weather_df.head()

,time,temperature_2m (°C),relative_humidity_2m (%),rain (mm),snowfall (cm),cloud_cover (%),wind_speed_10m (km/h),Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,2015-01-01 00:00:00,-4,49,0,0,0,14.1,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01 01:00:00,-4,49,0,0,0,14.1,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01 02:00:00,-4.2,52,0,0,0,14.7,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-01 03:00:00,-4.2,52,0,0,0,15.6,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-01 04:00:00,-4.2,50,0,0,0,16.1,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
print("Rain unique values:", weather_df["rain (mm)"].unique()[:10])
print("Snow unique values:", weather_df["snowfall (cm)"].unique()[:10])

Rain unique values: [0 0.2 0.8 1 1.3 1.5 2 1.4 2.1 1.1]
Snow unique values: [0 0.07 0.98 1.4 0.56 0.14 0.21 0.28 1.05 1.68]


In [27]:
weather_df = weather_df.loc[:, ~weather_df.columns.str.contains("^Unnamed")]

weather_df["time"] = pd.to_datetime(weather_df["time"], errors="coerce")

numeric_columns = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "rain (mm)",
    "snowfall (cm)",
    "cloud_cover (%)",
    "wind_speed_10m (km/h)"
]

for col in numeric_columns:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce")

weather_df = weather_df.dropna(subset=["time"]).reset_index(drop=True)

weather_df["year"] = weather_df["time"].dt.year
weather_df["month"] = weather_df["time"].dt.month
weather_df["day"] = weather_df["time"].dt.day
weather_df["hour"] = weather_df["time"].dt.hour
weather_df["dayofweek"] = weather_df["time"].dt.dayofweek
weather_df["is_weekend"] = weather_df["dayofweek"].isin([5,6]).astype(int)

In [28]:
def month_to_season(m):
    if m in [12,1,2]:
        return "winter"
    elif m in [3,4,5]:
        return "spring"
    elif m in [6,7,8]:
        return "summer"
    else:
        return "fall"

weather_df["season"] = weather_df["month"].apply(month_to_season)

In [29]:
weather_df["date"] = weather_df["time"].dt.date

In [31]:
weather_df = weather_df[[
    "date",
    "hour",
    "season",

    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "rain (mm)",
    "snowfall (cm)",
    "cloud_cover (%)",
    "wind_speed_10m (km/h)"
]]

In [32]:
weather_df.to_csv("dataset_weather.csv", index=False)

## selecting important columns

In [33]:
columns_to_keep = [
    'job__',
    'job_type',
    'borough',
    'gis_latitude',
    'gis_longitude',
    'pre__filing_date',
    'signoff_date'
]

construction_df = construction_df[
    [col for col in columns_to_keep if col in construction_df.columns]
]

In [35]:
# -----------------------------
# Clean job type
# -----------------------------
construction_df["job_type_clean"] = (
    construction_df["job_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# -----------------------------
# Group job types
# -----------------------------
def map_job_type(x):

    if x in ["A1", "A2", "A3"]:
        return "Alteration"

    elif x == "NB":
        return "New_Building"

    elif x == "DM":
        return "Demolition"

    else:
        return "Other"

construction_df["job_type_group"] = construction_df["job_type_clean"].apply(map_job_type)

# Check counts
print(construction_df["job_type_group"].value_counts())

job_type_group
Alteration      2368004
New_Building     199764
Demolition        80343
Other             66358
Name: count, dtype: int64


## renaming columns

In [37]:
construction_df = construction_df.rename(columns={
    'pre__filing_date': 'construction_start',
    'signoff_date': 'construction_end'
})

## datetime conversion

In [39]:
construction_df['construction_start'] = pd.to_datetime(
    construction_df['construction_start'],
    errors='coerce'
)
construction_df['construction_end'] = pd.to_datetime(
    construction_df['construction_end'],
    errors='coerce'
)

## removing rows without start date

In [40]:
construction_df = construction_df[construction_df['construction_start'].notna()]

## filling missing end dates

In [41]:
construction_df["construction_end"] = construction_df["construction_end"].fillna(construction_df["construction_start"])

In [43]:
construction_df["borough"] = construction_df["borough"].str.lower().str.strip()
construction_df["job_type_group"] = construction_df["job_type_group"].str.lower().str.strip()

## counting start events and end events

In [45]:
start_counts = (
    construction_df.groupby(["construction_start","borough"])
    .size()
    .reset_index(name="start_count")
)

end_counts = (
    construction_df.groupby(["construction_end","borough"])
    .size()
    .reset_index(name="end_count")
)

## rename to unified date

In [47]:
start_counts = start_counts.rename(columns={"construction_start":"date"})
end_counts = end_counts.rename(columns={"construction_end":"date"})

In [39]:
input_2 = 'This line combines the construction start events and construction end events into one table so you can later calculate how many construction projects are active on each date in each borough.'

In [49]:
construction_events = pd.merge(
    start_counts,
    end_counts,
    on=["date","borough"],
    how="outer"
).fillna(0)

## This gives number of active construction projects per borough per date\.

In [51]:
construction_events = construction_events.sort_values(["borough","date"])

construction_events["active_construction"] = (
    construction_events.groupby("borough")["start_count"].cumsum()
    - construction_events.groupby("borough")["end_count"].cumsum()
)

In [53]:
construction_events["construction_change"] = (
    construction_events.groupby("borough")["active_construction"]
    .diff()
)

In [55]:
construction_events["construction_change"] = (
    construction_events["construction_change"].fillna(0)
)

In [57]:
job_type_counts = (
    construction_df
    .groupby(["construction_start","borough","job_type_group"])
    .size()
    .reset_index(name="count")
)

In [45]:
input_7 = 'What we did (pivot / one-hot encoding)\n\nWe transformed it into this structure:\n\ndate	borough	demolition	alteration	new_building\n2019-01-01	brooklyn	2	1	0\n2019-01-02	queens	0	0	1\n\nThis means:\n\nnumber of jobs of each type on that day in that borough'

In [59]:
job_type_counts = job_type_counts.pivot_table(
    index=["construction_start","borough"],
    columns="job_type_group",
    values="count",
    fill_value=0
).reset_index()

job_type_counts = job_type_counts.rename(
    columns={"construction_start":"date"}
)

In [61]:
construction_events = construction_events.merge(
    job_type_counts,
    on=["date","borough"],
    how="left"
)

In [63]:
construction_events["borough"] = construction_events["borough"].str.lower().str.strip()

In [65]:
construction_events["date"] = pd.to_datetime(construction_events["date"])

In [67]:
construction_events = construction_events.sort_values(by="date", ascending=False)
construction_events.head(20)

,date,borough,start_count,end_count,active_construction,construction_change,alteration,demolition,new_building,other
37384,2026-03-23,staten island,4.0,7.0,0.0,-3.0,3.0,0.0,0.0,1.0
7335,2026-03-23,bronx,0.0,5.0,0.0,-5.0,NaN,NaN,NaN,NaN
22609,2026-03-23,manhattan,3.0,14.0,0.0,-11.0,3.0,0.0,0.0,0.0
15071,2026-03-23,brooklyn,3.0,3.0,0.0,0.0,2.0,0.0,0.0,1.0
30379,2026-03-23,queens,0.0,7.0,0.0,-7.0,NaN,NaN,NaN,NaN
15070,2026-03-22,brooklyn,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
22608,2026-03-20,manhattan,1.0,3.0,11.0,-2.0,1.0,0.0,0.0,0.0
15069,2026-03-20,brooklyn,4.0,9.0,0.0,-5.0,3.0,0.0,0.0,1.0
7334,2026-03-20,bronx,2.0,3.0,5.0,-1.0,1.0,0.0,0.0,1.0
30378,2026-03-20,queens,3.0,9.0,7.0,-6.0,3.0,0.0,0.0,0.0


traffic\_df\["time"\] = pd\.to\_datetime\(
    traffic\_df\["HH"\]\.astype\(str\) \+ ":" \+ traffic\_df\["MM"\]\.astype\(str\),
    format="%H:%M"
\)\.dt\.time

In [93]:
traffic_df.head()

,RequestID,Boro,Yr,M,D,HH,MM,traffic_volume,SegmentID,WktGeom,street,fromSt,toSt,Direction
0,22562,Queens,2016,5,8,8,45,260.0,155613,POINT (1059678.8154876027 198480.09766927382),HEMPSTEAD AVENUE,Cross Island Parkway,Cross Is Pkwy Nb En Hempstead Wb,WB
1,22562,Queens,2016,5,8,9,0,243.0,155613,POINT (1059678.8154876027 198480.09766927382),HEMPSTEAD AVENUE,Cross Island Parkway,Cross Is Pkwy Nb En Hempstead Wb,WB
2,22562,Queens,2016,5,8,9,15,245.0,155613,POINT (1059678.8154876027 198480.09766927382),HEMPSTEAD AVENUE,Cross Island Parkway,Cross Is Pkwy Nb En Hempstead Wb,WB
3,22562,Queens,2016,5,8,9,30,304.0,155613,POINT (1059678.8154876027 198480.09766927382),HEMPSTEAD AVENUE,Cross Island Parkway,Cross Is Pkwy Nb En Hempstead Wb,WB
4,22562,Queens,2016,5,8,9,45,312.0,155613,POINT (1059678.8154876027 198480.09766927382),HEMPSTEAD AVENUE,Cross Island Parkway,Cross Is Pkwy Nb En Hempstead Wb,WB


In [69]:
traffic_df["Vol"] = pd.to_numeric(
    traffic_df["Vol"],
    errors="coerce"
)

In [71]:
traffic_df = traffic_df.rename(columns={
    "Vol": "traffic_volume"
})

In [73]:
traffic_df["date"] = pd.to_datetime(traffic_df[["Yr", "M", "D"]].rename(columns={"Yr": "year", "M": "month", "D": "day"}))

In [75]:
# Crash dataset
crash_df["day_of_week"] = pd.to_datetime(crash_df["date"]).dt.dayofweek

# Traffic dataset
traffic_df["day_of_week"] = pd.to_datetime(traffic_df["date"]).dt.dayofweek

In [77]:
traffic_df = traffic_df.rename(columns={"Boro": "BOROUGH"})


start_date = pd.Timestamp(2015,1,1)
end_date = pd.Timestamp(2025,12,31)

traffic_df = traffic_df[
    (traffic_df["date"] >= start_date) &
    (traffic_df["date"] <= end_date)
]

traffic_df["hour"] = traffic_df["HH"]


traffic_df["BOROUGH"] = traffic_df["BOROUGH"].str.lower().str.strip()


traffic_borough_hour = (
    traffic_df
    .groupby(["BOROUGH", "hour", "day_of_week"])["traffic_volume"]
    .mean()
    .reset_index()
)


In [56]:
input_3 = 'Traffic sensors don\'t exist every day, so matching:\n\ndate + hour + borough\n\ncauses many NaN values.\n\nUsing:\n\nborough + hour traffic pattern\n\nremoves most NaNs and still captures traffic intensity patterns.'

In [79]:
crash_df["date"] = crash_df["CRASH DATE"].dt.floor("D")
crash_df["year"] = crash_df["CRASH DATE"].dt.year
crash_df["MONTH"] = crash_df["CRASH DATE"].dt.month
crash_df["hour"] = crash_df["CRASH TIME"].dt.hour

In [81]:
crash_df["date"] = pd.to_datetime(crash_df["date"])
weather_df["date"] = pd.to_datetime(weather_df["date"])

In [82]:
weather_df = weather_df.drop_duplicates(subset=["date","hour"])

In [84]:
traffic_borough_hour.rename(columns={"DATE": "date"}, inplace=True)

construction_events.rename(columns={
    "borough": "BOROUGH"
}, inplace=True)

In [111]:
print("CRASH DATASET COLUMNS:")
print(crash_df.columns)

print("\nCRASH DATASET HEAD:")
print(crash_df[["date", "BOROUGH"]].head())

print("\nWEATHER DATASET COLUMNS:")
print(weather_df.columns)

print("\nWEATHER DATASET HEAD:")
print(weather_df[["date", "hour"]].head())

print("\nTRAFFIC DATASET COLUMNS:")
print(traffic_borough_hour.columns)

print("\nTRAFFIC DATASET HEAD:")
print(traffic_borough_hour.head())

print("\nCONSTRUCTION DATASET COLUMNS:")
print(construction_events.columns)

print("\nCONSTRUCTION DATASET HEAD:")
print(construction_events.head())

print("\nDate column types:")
print("Crash date dtype:", crash_df["date"].dtype)
print("Weather date dtype:", weather_df["date"].dtype)
print("Construction date dtype:", construction_events["date"].dtype)

print("\nUnique borough values:")
print("Crash boroughs:", crash_df["BOROUGH"].unique())
print("Traffic boroughs:", traffic_borough_hour["BOROUGH"].unique())
print("Construction boroughs:", construction_events["BOROUGH"].unique())

CRASH DATASET COLUMNS:
Index(['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE',
       'LONGITUDE', 'LOCATION', 'NUMBER OF PERSONS INJURED',
       'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED',
       'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED',
       'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED',
       'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1',
       'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3',
       'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3',
       'HOUR', 'date', 'day_of_week', 'year', 'MONTH', 'hour'],
      dtype='object')

CRASH DATASET HEAD:
        date   BOROUGH
0 2023-11-01  brooklyn
1 2021-09-11  brooklyn
2 2021-12-14  brooklyn
3 2021-12-14  brooklyn
4 2021-12-14     bronx

WEATHER DATASET COLUMNS:
Index(['date', 'hour', 'season', 'temperature_2m (°C)',
       'relative_humidity_2m (%)', 'rain (mm)', 'snowfall (cm)',
       'cloud_cover (%)', 'wind_spe

In [62]:
input_5 = 'already done: \ncrash_df["date"] = pd.to_datetime(crash_df["date"])\nweather_df["date"] = pd.to_datetime(weather_df["date"])\ntraffic_borough_daily["date"] = pd.to_datetime(traffic_borough_daily["date"])\nconstruction_events["date"] = pd.to_datetime(construction_events["date"])'

In [83]:
print(crash_df["date"].dtype)
print(weather_df["date"].dtype)
print(construction_events["date"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [85]:
construction_events.columns

Index(['date', 'BOROUGH', 'start_count', 'end_count', 'active_construction',
       'construction_change', 'alteration', 'demolition', 'new_building',
       'other'],
      dtype='object')

In [86]:
crash_df["hour"] = crash_df["CRASH TIME"].dt.hour
traffic_borough_hour["hour"] = traffic_borough_hour["hour"].astype(int)

In [88]:
crash_df = crash_df.drop(columns=["HOUR"])

In [90]:
print(len(crash_df))
print(len(traffic_borough_hour))

1599858
840


In [92]:
weather_df["date"] = pd.to_datetime(weather_df["date"])

merged_df = crash_df.merge(
    weather_df,
    on=["date","hour"],
    how="left")
    

In [94]:
merged_df = merged_df.merge(
    traffic_borough_hour,
    on=["day_of_week","hour","BOROUGH"],
    how="left"
)


In [96]:
merged_df = merged_df.merge(
    construction_events,
    on=["date","BOROUGH"],
    how="left"
)

In [98]:
merged_df[["traffic_volume","active_construction"]].isna().mean()

traffic_volume         0.000000
active_construction    0.093105
dtype: float64

In [100]:
merged_df["active_construction"] = merged_df["active_construction"].fillna(0)

In [72]:
print(merged_df.shape)


merged_df.info()

(1599858, 41)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599858 entries, 0 to 1599857
Data columns (total 41 columns):
 #   Column                         Non-Null Count    Dtype         
---  ------                         --------------    -----         
 0   CRASH DATE                     1599858 non-null  datetime64[ns]
 1   CRASH TIME                     1599858 non-null  datetime64[ns]
 2   BOROUGH                        1599858 non-null  object        
 3   ZIP CODE                       1159828 non-null  object        
 4   LATITUDE                       1562065 non-null  float64       
 5   LONGITUDE                      1562065 non-null  float64       
 6   LOCATION                       1562065 non-null  object        
 7   NUMBER OF PERSONS INJURED      1599858 non-null  float64       
 8   NUMBER OF PERSONS KILLED       1599858 non-null  float64       
 9   NUMBER OF PEDESTRIANS INJURED  1599858 non-null  int64         
 10  NUMBER OF PEDESTRIANS KILLED   1599858 n

In [102]:
merged_df["start_count"] = merged_df["start_count"].fillna(0)
merged_df["end_count"] = merged_df["end_count"].fillna(0)
merged_df["construction_change"] = merged_df["construction_change"].fillna(0)

In [106]:
merged_df = merged_df.sort_values(by="date", ascending=False)
merged_df.head(20)

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,...,wind_speed_10m (km/h),traffic_volume,start_count,end_count,active_construction,construction_change,alteration,demolition,new_building,other
1596460,2025-12-31,1900-01-01 20:45:00,queens,NaN,40.716133,-73.818500,"(40.716133, -73.8185)",0.0,0.0,0,...,10.5,109.333672,4.0,12.0,339.0,-8.0,2.0,0.0,0.0,2.0
1596484,2025-12-31,1900-01-01 11:33:00,brooklyn,11213.0,40.668617,-73.945045,"(40.668617, -73.945045)",0.0,0.0,0,...,16.2,115.431124,1.0,10.0,547.0,-9.0,1.0,0.0,0.0,0.0
1596468,2025-12-31,1900-01-01 17:15:00,queens,11418.0,40.696865,-73.837166,"(40.696865, -73.837166)",0.0,0.0,0,...,9.4,156.304348,4.0,12.0,339.0,-8.0,2.0,0.0,0.0,2.0
1596471,2025-12-31,1900-01-01 10:00:00,staten island,10306.0,40.574257,-74.106430,"(40.574257, -74.10643)",0.0,0.0,0,...,13.6,154.798295,0.0,2.0,111.0,-2.0,NaN,NaN,NaN,NaN
1596457,2025-12-31,1900-01-01 13:50:00,queens,11355.0,40.756800,-73.817070,"(40.7568, -73.81707)",0.0,0.0,0,...,16.3,130.101124,4.0,12.0,339.0,-8.0,2.0,0.0,0.0,2.0
1596472,2025-12-31,1900-01-01 14:07:00,brooklyn,11211.0,40.717834,-73.957726,"(40.717834, -73.957726)",0.0,0.0,0,...,16.5,126.106272,1.0,10.0,547.0,-9.0,1.0,0.0,0.0,0.0
1596474,2025-12-31,1900-01-01 22:01:00,manhattan,10031.0,40.829082,-73.941080,"(40.829082, -73.94108)",1.0,0.0,1,...,11.4,161.792818,1.0,5.0,679.0,-4.0,1.0,0.0,0.0,0.0
1596404,2025-12-31,1900-01-01 02:30:00,bronx,10468.0,40.862194,-73.909380,"(40.862194, -73.90938)",0.0,0.0,0,...,14.0,21.868677,1.0,3.0,142.0,-2.0,1.0,0.0,0.0,0.0
1596477,2025-12-31,1900-01-01 19:13:00,bronx,10467.0,40.885693,-73.861640,"(40.885693, -73.86164)",1.0,0.0,1,...,9.7,125.597858,1.0,3.0,142.0,-2.0,1.0,0.0,0.0,0.0
1596478,2025-12-31,1900-01-01 06:38:00,brooklyn,11210.0,40.629055,-73.948326,"(40.629055, -73.948326)",0.0,0.0,0,...,5.0,101.932565,1.0,10.0,547.0,-9.0,1.0,0.0,0.0,0.0


In [121]:
print("Min date:", merged_df['year'].min())
print("Max date:", merged_df['year'].max())

Min date: 2015
Max date: 2025


In [108]:
merged_df.groupby("BOROUGH")[["traffic_volume","active_construction"]].apply(
    lambda x: x.isna().mean()
)

,traffic_volume,active_construction
BOROUGH,,
bronx,0.0,0.0
brooklyn,0.0,0.0
manhattan,0.0,0.0
queens,0.0,0.0
staten island,0.0,0.0


In [77]:
merged_df.isna().mean().sort_values(ascending=False).head(10)

VEHICLE TYPE CODE 3              0.930263
CONTRIBUTING FACTOR VEHICLE 3    0.926387
ZIP CODE                         0.275043
VEHICLE TYPE CODE 2              0.236255
CONTRIBUTING FACTOR VEHICLE 2    0.176405
new_building                     0.113499
demolition                       0.113499
alteration                       0.113499
other                            0.113499
LATITUDE                         0.023623
dtype: float64

In [110]:
crash_df.to_csv("dataset_crash.csv", index=False)

In [112]:
weather_df.to_csv("dataset_weather.csv", index=False)

In [114]:
construction_events.to_csv(
    "dataset_construction.csv",
    index=False
)

In [116]:
traffic_df.to_csv(
    "dataset_traffic.csv",
    index=False
)

In [119]:
traffic_borough_hour.to_csv(
    "traffic_pattern_120.csv",
    index=False
)

In [118]:
merged_df.to_csv(
    "dataset_merged.csv",
    index=False
)